In [1]:
"""
HyenaDNA-inspired Model: Masked Nucleotide Prediction
------------------------------------------------------
Dataset: Salmonella Typhimurium (6 genomes)

Input files:
- train.txt : 5 genomes, raw sequences (training)
- test.txt  : Salmonella 6, raw sequences (testing)

Masking Strategy:
- Same 5-mer rotation masking at 20% (1 in every 5 bases)
- Masked positions replaced with MASK token
- Masking applied on the fly inside the dataset class

Model:
- Custom Hyena operator implemented from scratch in PyTorch
- Same 5-mer tokenisation as transformer_model_2.ipynb
- Hyena replaces attention with implicit long convolutions

Comparison vs transformer_model_2.ipynb:
- Same task, same data, same 5-mer vocabulary, same evaluation
- Same parameter count target (~440K)
- Only difference: Hyena operators vs Multi-head Attention

Training:
- Optimiser : AdamW
- Loss      : Cross Entropy (masked positions only)
- Device    : CUDA if available, else CPU
"""

'\nHyenaDNA-inspired Model: Masked Nucleotide Prediction\n------------------------------------------------------\nDataset: Salmonella Typhimurium (6 genomes)\n\nInput files:\n- train.txt : 5 genomes, raw sequences (training)\n- test.txt  : Salmonella 6, raw sequences (testing)\n\nMasking Strategy:\n- Same 5-mer rotation masking at 20% (1 in every 5 bases)\n- Masked positions replaced with MASK token\n- Masking applied on the fly inside the dataset class\n\nModel:\n- Custom Hyena operator implemented from scratch in PyTorch\n- Same 5-mer tokenisation as transformer_model_2.ipynb\n- Hyena replaces attention with implicit long convolutions\n\nComparison vs transformer_model_2.ipynb:\n- Same task, same data, same 5-mer vocabulary, same evaluation\n- Same parameter count target (~440K)\n- Only difference: Hyena operators vs Multi-head Attention\n\nTraining:\n- Optimiser : AdamW\n- Loss      : Cross Entropy (masked positions only)\n- Device    : CUDA if available, else CPU\n'

Step 1 - Imports

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import itertools
from torch.utils.data import Dataset, DataLoader

Step 2 - Hyperparameters and Configuration

In [7]:
# Build 5-mer vocabulary
bases    = ['A', 'C', 'G', 'T']
kmers    = [''.join(k) for k in itertools.product(bases, repeat=5)]
kmer2idx = {kmer: idx for idx, kmer in enumerate(kmers)}

kmer2idx['MASK'] = 1024
kmer2idx['PAD']  = 1025
kmer2idx['UNK']  = 1026

VOCAB      = kmer2idx
VOCAB_SIZE = len(VOCAB)  # 1027

# Windowing
WINDOW_SIZE   = 128    # 5-mer windows
STRIDE        = 64

# Model Architecture
EMBED_DIM     = 128    # embedding dimension
NUM_LAYERS    = 2     # number of Hyena blocks
ORDER         = 1      # Hyena operator order
DROPOUT       = 0.1

# Training
BATCH_SIZE    = 64
EPOCHS        = 20
LEARNING_RATE = 3e-4

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Vocabulary size : {VOCAB_SIZE}")
print(f"  Window size     : {WINDOW_SIZE}")
print(f"  Device          : {DEVICE}")

  Vocabulary size : 1027
  Window size     : 128
  Device          : cuda


Step 3 - Load Dataset

In [ ]:
class DNADataset(Dataset):
    def __init__(self, path, window_size=WINDOW_SIZE, stride=STRIDE):
        self.windows_input  = []
        self.windows_target = []

        original_seqs = open(path, "r").readlines()

        for seq in original_seqs:
            seq = seq.strip().upper()

            # Convert sequence to 5-mer tokens
            kmers = []
            for i in range(0, len(seq) - 4, 1):
                kmer = seq[i:i+5]
                if 'N' in kmer:
                    kmers.append(VOCAB['UNK'])
                else:
                    kmers.append(VOCAB.get(kmer, VOCAB['UNK']))

            # Mask every 5th kmer (rotation masking)
            masked_kmers = kmers.copy()
            for i in range(len(masked_kmers)):
                if i % 5 == (i // 5) % 5:
                    masked_kmers[i] = VOCAB['MASK']

            # Slice into windows
            for start in range(0, len(kmers) - window_size, stride):
                input_window  = masked_kmers[start : start + window_size]
                target_window = kmers       [start : start + window_size]

                self.windows_input.append(torch.tensor(input_window,  dtype=torch.long))
                self.windows_target.append(torch.tensor(target_window, dtype=torch.long))

    def __len__(self):
        return len(self.windows_input)

    def __getitem__(self, idx):
        return self.windows_input[idx], self.windows_target[idx]


# Load datasets
print("Loading training set...")
train_dataset = DNADataset("train.txt")
print("Loading test set...")
test_dataset  = DNADataset("test.txt")

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"  Training windows : {len(train_dataset):,}")
print(f"  Test windows     : {len(test_dataset):,}")

Loading training set...
Loading test set...
  Training windows : 384,028
  Test windows     : 75,896


Step 4 - Hyena Model

In [ ]:
class HyenaOperator(nn.Module):
    """
    Simplified Hyena operator — replaces attention with FFT-based 
    convolution which is subquadratic and handles long sequences better.
    """
    def __init__(self, embed_dim, max_len=WINDOW_SIZE, order=ORDER, dropout=DROPOUT):
        super().__init__()
        self.embed_dim = embed_dim
        self.order     = order

        # Input projection — projects input into order+1 streams
        self.input_proj = nn.Linear(embed_dim, (order + 1) * embed_dim)

        # Learnable convolution filters — one per order
        # These are the implicit long convolutions that replace attention
        self.filters = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, 32),
                nn.SiLU(),
                nn.Linear(32, embed_dim)
            )
            for _ in range(order)
        ])

        # Position encoding for filter generation
        positions = torch.linspace(0, 1, max_len).unsqueeze(1)  # (max_len, 1)
        self.register_buffer('positions', positions)

        self.output_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch, seq_len, embed_dim)
        batch, seq_len, _ = x.shape

        # Project input into order+1 streams
        projected = self.input_proj(x)  # (batch, seq_len, (order+1)*embed_dim)
        streams   = projected.split(self.embed_dim, dim=-1)  # list of order+1 tensors

        # v is the value stream, h_inputs are the filter streams
        v        = streams[0]
        h_inputs = streams[1:]

        # Apply Hyena recurrence
        # Each order applies a convolution then gates with next stream
        out = v
        pos = self.positions[:seq_len]  # (seq_len, 1)

        for i in range(self.order):
            # Generate filter from positions
            filt = self.filters[i](pos)              # (seq_len, embed_dim)
            filt = filt.unsqueeze(0).expand(batch, -1, -1)  # (batch, seq_len, embed_dim)

            # FFT convolution
            x_fft    = torch.fft.rfft(out,   n=2*seq_len, dim=1)
            filt_fft = torch.fft.rfft(filt,  n=2*seq_len, dim=1)
            out      = torch.fft.irfft(x_fft * filt_fft, n=2*seq_len, dim=1)[:, :seq_len, :]

            # Gate with corresponding stream
            out = out * h_inputs[i]

        out = self.output_proj(out)
        out = self.dropout(out)
        return out


class HyenaBlock(nn.Module):
    """Single Hyena block — Hyena operator + feedforward + layer norms"""
    def __init__(self, embed_dim, ff_dim=256, dropout=DROPOUT):
        super().__init__()
        self.norm1  = nn.LayerNorm(embed_dim)
        self.norm2  = nn.LayerNorm(embed_dim)
        self.hyena  = HyenaOperator(embed_dim)
        self.ff     = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # Hyena with residual connection
        x = x + self.hyena(self.norm1(x))
        # Feedforward with residual connection
        x = x + self.ff(self.norm2(x))
        return x


class HyenaDNAModel(nn.Module):
    """Full Hyena model for masked 5-mer prediction"""
    def __init__(self):
        super().__init__()

        # 5-mer embedding
        self.embedding     = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=VOCAB['PAD'])
        self.pos_embedding = nn.Embedding(WINDOW_SIZE, EMBED_DIM)

        # Stack of Hyena blocks
        self.blocks = nn.ModuleList([
            HyenaBlock(EMBED_DIM) for _ in range(NUM_LAYERS)
        ])

        self.norm         = nn.LayerNorm(EMBED_DIM)
        self.output_layer = nn.Linear(EMBED_DIM, VOCAB_SIZE)

    def forward(self, x):
        # x shape: (batch, window_size)
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        x = self.output_layer(x)
        return x  # (batch, window_size, vocab_size)


# Initialise model
model = HyenaDNAModel().to(DEVICE)
print(model)
print(f"\n  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

HyenaDNAModel(
  (embedding): Embedding(1027, 128, padding_idx=1025)
  (pos_embedding): Embedding(128, 128)
  (blocks): ModuleList(
    (0-1): 2 x HyenaBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (hyena): HyenaOperator(
        (input_proj): Linear(in_features=128, out_features=256, bias=True)
        (filters): ModuleList(
          (0): Sequential(
            (0): Linear(in_features=1, out_features=32, bias=True)
            (1): SiLU()
            (2): Linear(in_features=32, out_features=128, bias=True)
          )
        )
        (output_proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=256, out_features=128,